# ITO5201 – Assessment 1: Section 2
## Probability
**Student:** Johannes Coetzee  
**Student Number:** 36384852

In [1]:
import numpy as np

# DEBUG=True prints every simulated draw inside fruit_experiment — fine for the
# 4-repetition demo, but it embeds ~200,000 lines of output in the 100,000-rep
# verification cell. Keep False for submission; the demo cell prints its result
# arrays explicitly either way.
DEBUG = False # Set to True to enable debug mode, False to disable

---
## Question 4 – Bayes Rule
### Q4.I – Simulate the Box/Fruit Experiment

Box contents:
- **Red**: 3 apples, 5 oranges
- **Blue**: 4 apples, 4 oranges
- **Yellow**: 1 apple, 1 orange

In [2]:
# DEFINITION: Box class to represent a box containing fruits

class Box:

    """
    Class representing a box that can contain different types of fruits. 
    Each fruit type has an associated weight (probability) for sampling.

    Parameters:
    __init__(self, name): Initializes a Box instance with a given name. The box starts empty.
    - name: The name of the box (string).
    
    sample_fruit(self, rng): Samples a fruit from the box based on the weights of the fruit types.
    - rng: A random number generator (e.g., np.random.default_rng()).
    
    add_fruit(self, fruit, count=1): Adds a specified number of fruits of a given type to the box.
    - fruit: The type of fruit to add (string).
    - count: The number of fruits to add (default is 1).

    remove_fruit(self, fruit, count=1): Removes a specified number of fruits of a given type from the box.
    - fruit: The type of fruit to remove (string).
    - count: The number of fruits to remove (default is 1). Raises ValueError if there are not enough fruits of that type in the box.

    NumPy functions used:
    - np.array: Creates a new array.
    - np.sum: Computes the sum of array elements.
    - np.random.default_rng: Creates a new random number generator.
    Ref: https://numpy.org/doc/stable/reference/generated/numpy.array.html
    Ref: https://numpy.org/doc/stable/reference/generated/numpy.sum.html
    Ref: https://numpy.org/doc/stable/reference/random/generator.html
    """


    def __init__(self, name):
        self.name = name
        self.total_fruits = 0
        self.fruit_types = []
        self.contents = {}
        self.fruit_weights = np.array([])

    def sample_fruit(self, rng) -> str:
        return rng.choice(self.fruit_types, p=self.fruit_weights)
    
    def add_fruit(self, fruit, count=1):
        if fruit in self.contents:
            self.contents[fruit] += count
        else:
            self.contents[fruit] = count
        self.total_fruits = sum(self.contents.values())
        self.fruit_types = list(self.contents.keys())
        self.fruit_weights = np.array(list(self.contents.values())) / self.total_fruits


    # This function is not required for BAYES
    def remove_fruit(self, fruit, count=1):
        if fruit in self.contents and self.contents[fruit] >= count:
            self.contents[fruit] -= count
            if self.contents[fruit] == 0:
                del self.contents[fruit]
            self.total_fruits = sum(self.contents.values())
            self.fruit_types = list(self.contents.keys())
            self.fruit_weights = np.array(list(self.contents.values())) / self.total_fruits
        else:
            raise ValueError(f"Not enough {fruit} to remove.")
        
    def get_fruit_count(self, fruit) -> int:
        return self.contents.get(fruit, 0)



In [3]:
# SETUP: Create boxes and add fruits
"""
This cell sets up three boxes (red, blue, and yellow) and adds different types of fruits (apples and oranges) to each box.
"""


red_box = Box("red")
blue_box = Box("blue")
yellow_box = Box("yellow")
    
red_box.add_fruit("apple", 3)
red_box.add_fruit("orange", 5)
blue_box.add_fruit("apple", 4)
blue_box.add_fruit("orange", 4)
yellow_box.add_fruit("apple", 1)
yellow_box.add_fruit("orange", 1)

print(f"Red box contents: {red_box.contents}") if DEBUG else None
print(f"Blue box contents: {blue_box.contents}") if DEBUG else None
print(f"Yellow box contents: {yellow_box.contents}") if DEBUG else None


In [4]:
# FUNCTION: Simulate selecting a box and a fruit

def fruit_experiment(n, rng=None):
    """
    Simulate selecting a box uniformly at random, then a fruit from that box.
    Returns arrays of selected boxes and fruits for n repetitions.

    Source: the random sampling pattern (np.random.default_rng and weighted
    Generator.choice) follows Activity 0.1 (Modules/0/Activity.0.1.ipynb,
    "Generating Random Data", Task B).

    Parameters:
    - n: number of repetitions
    - rng: optional random number generator (default is None, which uses np.random.default_rng

    Returns:
    - boxes: array of selected box names
    - fruits: array of selected fruit names
    """

    if rng is None:
        rng = np.random.default_rng()

    boxes = []
    fruits = []
    box_list = [red_box, blue_box, yellow_box]
    for _ in range(n):
        chosen_box = rng.choice(box_list)
        print(f"Chosen box: {chosen_box.name}") if DEBUG else None
        boxes.append(chosen_box.name)
        fruits.append(chosen_box.sample_fruit(rng))
        print(f"Chosen fruit: {fruits[-1]}") if DEBUG else None
    return np.array(boxes), np.array(fruits)


In [5]:
# EXECUTION: Run the fruit experiment with 4 repetitions and a fixed random seed

"""
This cell runs the fruit experiment with 4 repetitions using a fixed random seed for reproducibility.
It creates a random number generator with a seed of 42, then calls the fruit_experiment 
function to simulate the selection of boxes and fruits. The results, including the selected boxes and fruits,
NumPy functions used:
- np.random.default_rng: Creates a new random number generator.
Ref: https://numpy.org/doc/stable/reference/random/generator.html
"""

FRUIT_USE_RANDOM = True  # Set to True to use random number generation, False to disable

rng = np.random.default_rng(42) if FRUIT_USE_RANDOM else None  # Use default RNG if randomization is disabled
boxes, fruits = fruit_experiment(4, rng)
print(boxes)
print(fruits)


['red' 'yellow' 'red' 'yellow']
['orange' 'orange' 'apple' 'orange']


### Q4.II – Derive P(yellow | apple) Using Bayes' Rule
**Formal derivation:**

Let $B \in \{\text{red}, \text{blue}, \text{yellow}\}$ denote the selected box and $F \in \{\text{apple}, \text{orange}\}$ the selected fruit.

**Prior** (uniform box selection):
$$P(B = b) = \frac{1}{3} \quad \text{for all } b$$

**Likelihoods** $P(F = \text{apple} \mid B = b)$:

$$P(F = \text{apple} \mid B = \text{red}) = \frac{3}{8}$$

$$P(F = \text{apple} \mid B = \text{blue}) = \frac{4}{8} = \frac{1}{2}$$

$$P(F = \text{apple} \mid B = \text{yellow}) = \frac{1}{2}$$

**Marginal** $P(F = \text{apple})$ via total probability theorem:

$$P(F = \text{apple}) = \sum_{b} P(F = \text{apple} \mid B = b) \cdot P(B = b)$$

$$= \frac{3}{8} \cdot \frac{1}{3} + \frac{1}{2} \cdot \frac{1}{3} + \frac{1}{2} \cdot \frac{1}{3}$$

$$= \frac{3}{24} + \frac{4}{24} + \frac{4}{24} = \frac{11}{24}$$

**Posterior** via Bayes' Rule:
$$P(B = \text{yellow} \mid F = \text{apple}) = \frac{P(F = \text{apple} \mid B = \text{yellow}) \cdot P(B = \text{yellow})}{P(F = \text{apple})}$$

$$= \frac{\dfrac{1}{2} \cdot \dfrac{1}{3}}{\dfrac{11}{24}} = \frac{\dfrac{1}{6}}{\dfrac{11}{24}} = \frac{1}{6} \cdot \frac{24}{11} = \frac{4}{11} \approx 0.364$$

### Verification Against Simulation

In [6]:
# EXECUTION: Run the fruit experiment with 100,000 repetitions and a fixed random seed

"""
This cell runs the fruit experiment with 100,000 repetitions using a fixed random seed for reproducibility.
It creates a random number generator with a seed of 0, then calls the fruit_experiment function to simulate the selection of boxes and fruits.
The results, including the selected boxes and fruits,

NumPy functions used:
- np.random.default_rng: Creates a new random number generator.
Ref: https://numpy.org/doc/stable/reference/random/generator.html
- np.where: Returns the indices of elements in an input array that satisfy a given condition.
Ref: https://numpy.org/doc/stable/reference/generated/numpy.where.html
- np.sum: Computes the sum of array elements over a given axis.
Ref: https://numpy.org/doc/stable/reference/generated/numpy.sum.html

"""

# Verify P(yellow | apple) derivation against simulation
rng = np.random.default_rng(0) if FRUIT_USE_RANDOM else np.random.default_rng()  # Use default RNG if randomization is disabled
boxes, fruits = fruit_experiment(100_000, rng)

# Empirical P(yellow | apple)
apple_indices      = np.where(fruits == "apple")[0]
yellow_given_apple = np.sum(boxes[apple_indices] == "yellow") / len(apple_indices)

print(f"Empirical  P(yellow | apple) : {yellow_given_apple:.4f}")
print(f"Analytical P(yellow | apple) : {4/11:.4f}  (= 4/11)")
print(f"Difference                   : {abs(yellow_given_apple - 4/11):.4f}  "
      f"({'within' if abs(yellow_given_apple - 4/11) < 0.01 else 'outside'} expected statistical margin)")


Empirical  P(yellow | apple) : 0.3650
Analytical P(yellow | apple) : 0.3636  (= 4/11)
Difference                   : 0.0014  (within expected statistical margin)


---
## Question 5 – Expected Values
### Q5.I – Implement `die_experiment`

**Game rules:**
- $X$ = outcome of first die roll (1–6)
- $Y(i)$ = outcome of $i$-th subsequent roll if $i \leq X$, else 0
- $Z = \sum_{i=1}^{6} Y(i)$ = player score

In [7]:
# DEFINITION: Simulate the one-player die game

DIE_USE_RANDOM = True  # Set to True to use random number generation, False to disable

def die_experiment(n_repetitions, rng=None) -> np.ndarray:
    """
    Simulate the one-player die game for n_repetitions rounds.

    Source: the rng pattern (np.random.default_rng / Generator.integers) follows
    Activity 0.1 (Modules/0/Activity.0.1.ipynb, "Generating Random Data");
    estimating an expected value from repeated simulation follows Activity 0.2
    (Modules/0/Activity.0.2.ipynb, "Expected Value", Task D).
    
    Parameters:
    n_repetitions (int): The number of times to repeat the die game.
    rng (np.random.Generator, optional): A random number generator. If None, a default generator is created.

    Returns:
    np.ndarray: An array containing the scores from each repetition of the die game.

    """
    if rng is None:
        rng = np.random.default_rng()
    
    scores = []
    for _ in range(n_repetitions):
        X = rng.integers(1, 7)           # first roll
        Z = rng.integers(1, 7, size=X).sum()  # sum of X subsequent rolls
        scores.append(Z)
    return np.array(scores)


In [8]:
# EXECUTION: Run the die experiment with 10 repetitions and a fixed random seed
"""
This cell runs the die experiment with 10 repetitions using a fixed random seed for reproducibility.
It creates a random number generator with a seed of 42, then calls the die_experiment function to simulate the die game. 
The results, including the scores from each repetition, are printed.
"""

# Initialize the random number generator with a fixed seed for reproducibility
rng = np.random.default_rng(42) if DIE_USE_RANDOM else None  # Use default RNG if randomization is disabled
# Example usage of the die_experiment function
print("Die experiment results for 10 repetitions:")
scores = die_experiment(10, rng)
print(f"Scores: {scores}")


Die experiment results for 10 repetitions:
Scores: [ 5 13 18 21  9 24  7 19 18 23]


### Q5.II – Estimate E[Z] with 95% Confidence Interval

In [9]:
# EXECUTION: Run the die experiment with 10,000 repetitions and compute mean and 95% confidence interval
"""
In this cell we leverage the following numpy functions to compute the mean and 95% confidence interval of the scores obtained from the die_experiment function:
np.mean: Computes the arithmetic mean of the scores.
np.std: Computes the standard deviation of the scores.

Numpy is used to efficiently handle the array of scores and perform 
statistical calculations. 
The 95% confidence interval is calculated using the formula: 
mean ± 1.96 * (std / sqrt(n)), where n is the number of repetitions. 
This provides an estimate of the range within which we can be 95% confident 
that the true mean score lies.
The value of 1.96 is derived from the standard normal distribution, corresponding to a 95% confidence level.
ref: LawrenceSeminarioRomero [CC BY-SA 4.0]. Wikimedia Commons. https://commons.wikimedia.org/wiki/File:Normal_distribution_CDF.svg
Math and stat Cheatsheet. pg 16. Monash University. https://learning.monash.edu/pluginfile.php/7095552/mod_label/intro/FIT5197_2018S1_cheatsheet.pdf
"""

n = 10_000 # Number of repetitions for the die experiment
scores = die_experiment(n, rng)
mean_score = np.mean(scores)
std_dev = np.std(scores)  # Sample standard deviation
confidence_interval = 1.96 * std_dev / np.sqrt(n)
lower_bound = mean_score - confidence_interval
upper_bound = mean_score + confidence_interval
print(f"Mean score: {mean_score}")
print(f"95% Confidence Interval: [{lower_bound}, {upper_bound}]")


Mean score: 12.2706
95% Confidence Interval: [12.137016868063881, 12.404183131936119]


### Q5.III – Analytical Derivation of E[Z]

**Step 1: Conditional expectation $E[Z \mid X = x]$**

The expected value of a single fair six-sided die roll:

$$E[Y] = \sum_{k=1}^{6} k \cdot \frac{1}{6} = \frac{1 + 2 + 3 + 4 + 5 + 6}{6}
= \frac{21}{6} = \frac{7}{2}$$

Given $X = x$, exactly $x$ dice are rolled, each with expected value $\frac{7}{2}$, each with $E[Y(i)] = \frac{7}{2}$, and $Y(i) = 0$ for $i > x$:

$$E[Z \mid X = x] = \sum_{i=1}^{6} E[Y(i) \mid X = x] = \sum_{i=1}^{x} \frac{7}{2} + \sum_{i=x+1}^{6} 0 = \frac{7x}{2}$$

**Step 2: Marginal expectation via total expectation**

$$E[Z] = \sum_{x=1}^{6} E[Z \mid X = x] \cdot P(X = x) = \sum_{x=1}^{6} \frac{7x}{2} \cdot \frac{1}{6} = \frac{7}{12} \sum_{x=1}^{6} x$$

$$= \frac{7}{12} \cdot (1 + 2 + 3 + 4 + 5 + 6) = \frac{7}{12} \cdot 21 =
\frac{147}{12} = \frac{49}{4} = 12.25$$

So the expected player score is $E[Z] = \dfrac{49}{4} = 12.25$.